# Track C — Brysbaert Concreteness Scoring Experiment

**Goal**: Validate that average word concreteness of directive content words is a reliable specificity signal, before wiring it into `_evaluate_specificity`.

**Source**: Brysbaert et al. (2014) — 40,000 English word concreteness ratings (1=abstract, 5=concrete).  
Free download: http://crr.ugent.be/papers/Concreteness_ratings_Brysbaert_et_al_BRM.xlsx

**Method**:
1. Download and cache the Brysbaert database
2. Score 20 directive pairs (vague vs specific versions of the same task)
3. Validate that specific directives score higher than vague ones
4. Find optimal thresholds for the +0.10 / −0.08 scoring cutoffs
5. Conclusion: is mean concreteness a reliable specificity signal?

In [ ]:
import os, pathlib, urllib.request, re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams["figure.dpi"] = 120

CACHE_DIR = pathlib.Path.home() / ".cache" / "mycontext"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / "brysbaert2014.csv"

print("Cache location:", CACHE_PATH)

In [ ]:
# ── Download / load the Brysbaert database ────────────────────────────────────
# If the xlsx is not cached, try to download and convert to CSV.
# The xlsx is available from the supplementary materials of the paper.
# Multiple mirror URLs tried in order.

DOWNLOAD_URLS = [
    "http://crr.ugent.be/papers/Concreteness_ratings_Brysbaert_et_al_BRM.xlsx",
    "https://static-content.springer.com/esm/art%3A10.3758%2Fs13428-013-0403-5/MediaObjects/13428_2013_403_MOESM1_ESM.xlsx",
]

def download_brysbaert():
    """Download Brysbaert (2014) concreteness norms, cache as CSV."""
    xlsx_path = CACHE_DIR / "brysbaert2014.xlsx"
    for url in DOWNLOAD_URLS:
        try:
            print(f"Trying: {url}")
            urllib.request.urlretrieve(url, xlsx_path)
            print("  Downloaded OK")
            break
        except Exception as e:
            print(f"  Failed: {e}")
    else:
        return None  # all mirrors failed

    try:
        df = pd.read_excel(xlsx_path, engine="openpyxl")
        # Standardize column names across versions
        df.columns = [c.strip().lower() for c in df.columns]
        word_col = next((c for c in df.columns if "word" in c), df.columns[0])
        rating_col = next((c for c in df.columns if "conc" in c and "m" in c), None)
        if rating_col is None:
            rating_col = next((c for c in df.columns if "rating" in c or "mean" in c), df.columns[2])
        out = df[[word_col, rating_col]].rename(columns={word_col: "word", rating_col: "concreteness"})
        out["word"] = out["word"].str.lower().str.strip()
        out = out.dropna()
        out.to_csv(CACHE_PATH, index=False)
        print(f"  Saved CSV: {CACHE_PATH} ({len(out):,} words)")
        return out
    except Exception as e:
        print(f"  Parse error: {e}")
        return None


def load_brysbaert():
    """Load concreteness DB. Download if needed. Returns dict word->float."""
    if CACHE_PATH.exists():
        df = pd.read_csv(CACHE_PATH)
        print(f"Loaded from cache: {len(df):,} words")
    else:
        print("Cache not found — downloading...")
        df = download_brysbaert()
        if df is None:
            print("Download failed. Using built-in mini sample for demo.")
            # Fallback: 200-word built-in sample for demo
            MINI = {
                "dog": 4.93, "cat": 4.90, "table": 4.80, "chair": 4.78, "house": 4.72,
                "report": 3.62, "document": 3.55, "database": 3.20, "algorithm": 2.84,
                "system": 2.70, "process": 2.60, "framework": 2.52, "methodology": 2.45,
                "issue": 2.40, "problem": 2.38, "solution": 2.35, "performance": 2.30,
                "thing": 1.95, "stuff": 1.90, "aspect": 1.85, "notion": 1.72,
                "analysis": 2.25, "impact": 2.20, "value": 2.10, "concept": 1.95,
                "memory": 3.45, "cache": 3.30, "query": 3.25, "index": 3.20,
                "server": 3.85, "network": 3.60, "latency": 2.90, "throughput": 2.65,
                "metric": 2.78, "threshold": 2.72, "baseline": 2.68, "benchmark": 2.75,
                "deployment": 3.10, "authentication": 3.05, "encryption": 3.00,
                "revenue": 3.50, "profit": 3.40, "cost": 3.38, "budget": 3.62,
                "customer": 3.85, "churn": 3.42, "retention": 3.38, "cohort": 3.15,
                "quality": 2.15, "clarity": 2.08, "efficiency": 2.05, "relevance": 2.00,
            }
            return MINI
    return dict(zip(df["word"].str.lower(), df["concreteness"]))


CONCRETENESS_DB = load_brysbaert()
print(f"\nDatabase ready: {len(CONCRETENESS_DB):,} entries")
# Preview: most/least concrete
if isinstance(CONCRETENESS_DB, dict) and len(CONCRETENESS_DB) > 50:
    items = sorted(CONCRETENESS_DB.items(), key=lambda x: x[1])
    print(f"Most abstract (top 5): {items[:5]}")
    print(f"Most concrete (top 5): {items[-5:]}")

In [ ]:
# ── Concreteness scorer ───────────────────────────────────────────────────────

_STOPWORDS = {
    "the", "a", "an", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "are", "was", "were", "be", "been",
    "being", "have", "has", "had", "do", "does", "did", "will", "would",
    "could", "should", "may", "might", "shall", "can", "not", "no", "nor",
    "so", "yet", "both", "either", "neither", "each", "every", "all",
    "any", "few", "more", "most", "other", "some", "such", "than", "too",
    "very", "just", "that", "this", "these", "those", "it", "its", "our",
    "your", "their", "his", "her", "we", "you", "they", "i", "my", "your",
    "what", "which", "who", "how", "when", "where", "why", "if", "then",
    "please", "also", "use", "make", "give", "provide", "include", "ensure",
    "respond", "reply", "answer", "write", "create", "output", "format",
}


def score_concreteness(text: str, db: dict, min_word_len: int = 4) -> dict:
    """
    Score the average concreteness of content words in text.
    Returns dict with mean, matched_words, coverage, and interpretation.
    """
    words = re.findall(r"[a-z]+", text.lower())
    content_words = [w for w in words if len(w) >= min_word_len and w not in _STOPWORDS]
    matched = [(w, db[w]) for w in content_words if w in db]

    if len(matched) == 0:
        return {"mean": None, "matched": 0, "total_content": len(content_words),
                "coverage": 0.0, "words": [], "interpretation": "insufficient data"}

    mean_conc = np.mean([v for _, v in matched])
    coverage = len(matched) / max(len(content_words), 1)

    if mean_conc >= 3.5:
        interp = "very concrete"
    elif mean_conc >= 3.0:
        interp = "concrete"
    elif mean_conc >= 2.5:
        interp = "moderate"
    elif mean_conc >= 2.0:
        interp = "abstract"
    else:
        interp = "very abstract"

    return {
        "mean": round(mean_conc, 3),
        "matched": len(matched),
        "total_content": len(content_words),
        "coverage": round(coverage, 3),
        "words": matched[:10],  # top 10 for display
        "interpretation": interp,
    }


# Quick test
test_texts = [
    "Get the document",
    "Retrieve the Q3 2024 board report on supply chain risk from the SharePoint server",
    "Analyze things",
    "Identify the root cause of authentication latency increase in the payment gateway API",
]
for t in test_texts:
    r = score_concreteness(t, CONCRETENESS_DB)
    print(f"  [{r['mean']:.2f} / {r['interpretation']}] — {t[:70]}")

In [ ]:
# ── Test dataset: 20 directive pairs (vague vs specific) ─────────────────────
# Each pair tests the same task. Specific version should score higher.

DIRECTIVE_PAIRS = [
    ("Get the document",
     "Retrieve the Q3 2024 board report on supply chain risk from the SharePoint server"),
    ("Analyze the issue",
     "Analyze the 340ms authentication latency spike in the payment gateway API introduced in v2.4.1"),
    ("Improve performance",
     "Reduce database query latency by adding indexes on user_id and created_at columns in the orders table"),
    ("Write about customers",
     "Write a cohort retention analysis for enterprise customers who churned in Q2 2024 with ARR above $50k"),
    ("Do the analysis",
     "Perform root cause analysis of the 23% revenue decline in the EMEA region from January to March 2024"),
    ("Fix the problem",
     "Diagnose and fix the memory leak in the Node.js authentication service causing OOM crashes every 6 hours"),
    ("Create a plan",
     "Create a 90-day onboarding plan for a senior backend engineer joining the payments infrastructure team"),
    ("Explain the thing",
     "Explain the difference between B-tree and hash indexes in PostgreSQL and when to use each"),
    ("Review it",
     "Review the pull request for the JWT token refresh logic, focusing on security vulnerabilities and edge cases"),
    ("Summarize the stuff",
     "Summarize the key findings from the 2024 customer satisfaction survey across the enterprise and SMB segments"),
    ("Make recommendations",
     "Recommend 3 specific cost reduction strategies for the AWS infrastructure with projected monthly savings"),
    ("Look at the data",
     "Analyze the funnel drop-off rate at the checkout step using the Mixpanel cohort data from Q3 2024"),
    ("Consider the options",
     "Compare Redis Cluster vs Apache Kafka for real-time event streaming with throughput > 100k events/second"),
    ("Write something useful",
     "Write a post-mortem for the 4-hour outage on 2024-11-15 affecting the payments API with 99.2% error rate"),
    ("Handle it",
     "Triage the 47 open P1 bugs in Jira, categorize by root cause, and propose a resolution timeline by team"),
    ("Look into this",
     "Investigate the 3x increase in p99 latency for the /api/search endpoint after the Elasticsearch 8.9 upgrade"),
    ("Assess the situation",
     "Assess the compliance risk of storing EU customer PII in US-based S3 buckets under GDPR Article 46"),
    ("Give feedback",
     "Provide structured feedback on the Q4 OKR draft for the platform engineering team against SMART criteria"),
    ("Check the stuff",
     "Audit the IAM role assignments in AWS account 123456789 for overly permissive policies with * actions"),
    ("Do the work",
     "Build a SQL query to calculate 7-day rolling average DAU by country from the events table in BigQuery"),
]

print(f"Test pairs: {len(DIRECTIVE_PAIRS)}")
print("\nScoring all pairs...")
pair_results = []
for vague, specific in DIRECTIVE_PAIRS:
    vague_score  = score_concreteness(vague, CONCRETENESS_DB)
    spec_score   = score_concreteness(specific, CONCRETENESS_DB)
    pair_results.append({
        "vague": vague[:50],
        "specific": specific[:60],
        "vague_mean": vague_score["mean"],
        "spec_mean": spec_score["mean"],
        "delta": round((spec_score["mean"] or 0) - (vague_score["mean"] or 0), 3),
        "direction_correct": (spec_score["mean"] or 0) > (vague_score["mean"] or 0),
    })

pdf = pd.DataFrame(pair_results)
correct_direction = pdf["direction_correct"].mean()
print(f"\nDirection accuracy (specific > vague): {correct_direction:.1%} ({pdf['direction_correct'].sum()}/{len(pdf)} pairs)")
print(f"Mean delta (specific - vague): {pdf['delta'].mean():.3f}")
print(f"\nVague mean concreteness:    {pdf['vague_mean'].mean():.3f}")
print(f"Specific mean concreteness: {pdf['spec_mean'].mean():.3f}")

In [ ]:
# ── Visualization ────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: paired comparison
ax1 = axes[0]
x = range(len(pdf))
ax1.bar([i - 0.2 for i in x], pdf["vague_mean"], width=0.4, label="Vague", color="#e07b54", alpha=0.85)
ax1.bar([i + 0.2 for i in x], pdf["spec_mean"],  width=0.4, label="Specific", color="#4a90d9", alpha=0.85)
ax1.axhline(3.0, color="green", linestyle="--", linewidth=1, label="Threshold (3.0)")
ax1.axhline(2.0, color="red", linestyle="--", linewidth=1, label="Threshold (2.0)")
ax1.set_xticks(list(x))
ax1.set_xticklabels([f"P{i+1}" for i in x], fontsize=8)
ax1.set_ylabel("Mean Concreteness Score")
ax1.set_title("Concreteness: Vague vs Specific Directives")
ax1.legend(fontsize=9)
ax1.set_ylim(1.0, 5.0)

# Plot 2: delta distribution
ax2 = axes[1]
colors = ["#4a90d9" if d > 0 else "#e07b54" for d in pdf["delta"]]
ax2.bar(x, pdf["delta"], color=colors, alpha=0.85)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_xticks(list(x))
ax2.set_xticklabels([f"P{i+1}" for i in x], fontsize=8)
ax2.set_ylabel("Delta (Specific − Vague)")
ax2.set_title(f"Concreteness Lift per Pair\n(blue=correct direction, {pdf['direction_correct'].sum()}/{len(pdf)} correct)")
ax2.set_xlabel("Directive pair")

plt.tight_layout()
plt.savefig("../concreteness_pairs.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: research/concreteness_pairs.png")

In [ ]:
# ── Threshold calibration ─────────────────────────────────────────────────────
# Find the thresholds that maximally separate vague from specific directives.

all_vague    = [r["vague_mean"] for r in pair_results if r["vague_mean"]]
all_specific = [r["spec_mean"]  for r in pair_results if r["spec_mean"]]

print("Distribution stats:")
print(f"  Vague    — mean: {np.mean(all_vague):.3f}, std: {np.std(all_vague):.3f}, min: {min(all_vague):.3f}, max: {max(all_vague):.3f}")
print(f"  Specific — mean: {np.mean(all_specific):.3f}, std: {np.std(all_specific):.3f}, min: {min(all_specific):.3f}, max: {max(all_specific):.3f}")

# Test different threshold candidates
print("\nThreshold sensitivity (what % of vague score < threshold and specific score > threshold):")
for high_thresh in [2.5, 2.8, 3.0, 3.2, 3.5]:
    for low_thresh in [1.8, 2.0, 2.2, 2.5]:
        if low_thresh >= high_thresh:
            continue
        spec_above = sum(1 for v in all_specific if v >= high_thresh) / len(all_specific)
        vague_below = sum(1 for v in all_vague if v <= low_thresh) / len(all_vague)
        print(f"  high={high_thresh}, low={low_thresh}:  specific>{high_thresh}: {spec_above:.0%}  |  vague<{low_thresh}: {vague_below:.0%}")

# Recommended thresholds
print("\nRecommended thresholds for _evaluate_specificity:")
print("  mean_concreteness > 3.0  → +0.10 bonus (concrete directive)")
print("  mean_concreteness < 2.0  → −0.08 penalty (abstract directive)")
print("  coverage < 0.3           → skip scoring (too few words matched)")

## Conclusions

**Direction accuracy target**: ≥ 80% of pairs should show specific > vague.

**If direction accuracy ≥ 80%**: concreteness scoring is a reliable specificity signal — deploy in `_evaluate_specificity`.

**If direction accuracy < 80%**: either the DB download failed (using mini sample) or the signal is weaker than expected — investigate which pairs fail and whether they involve domain-specific jargon not in the general vocabulary.

**Threshold interpretation**:
- Mean concreteness > 3.0 = directive uses grounded, thing-like nouns → bonus
- Mean concreteness < 2.0 = directive uses abstract process/concept words → penalty
- Coverage < 30% = too few words matched, skip scoring to avoid noise

**Next**: If validated here, implement `_load_concreteness_db()` helper + concreteness scoring in `src/mycontext/intelligence/quality_metrics.py`